# RecoMart — Exploratory Data Analysis

This notebook explores the Retailrocket e-commerce dataset used in the RecoMart recommendation pipeline.

**Dataset:** Retailrocket (Kaggle) — ~2.7M events, ~400K items, ~1.4M visitors  
**Event types:** view, addtocart, transaction  
**Pseudo-ratings:** view=1, addtocart=2, transaction=3

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
print("Ready")

## 1. Load Raw Data

In [ ]:
from src.config import RAW_CSV_DIR
import os

# Find the latest date partition
events_dir = RAW_CSV_DIR / "events"
dates = sorted([d.name for d in events_dir.iterdir() if d.is_dir()]) if events_dir.exists() else []
dt = dates[-1] if dates else None
print(f"Using partition: {dt}")

# Load events
events = pd.read_csv(RAW_CSV_DIR / "events" / dt / "events.csv")
print(f"Events shape: {events.shape}")
events.head()

In [ ]:
# Load item properties
ip_dir = RAW_CSV_DIR / "item_properties" / dt
ip_files = sorted(ip_dir.glob("item_properties*.csv"))
item_props = pd.concat([pd.read_csv(f) for f in ip_files], ignore_index=True)
print(f"Item properties shape: {item_props.shape}")
item_props.head()

In [ ]:
# Load category tree
categories = pd.read_csv(RAW_CSV_DIR / "category_tree" / dt / "category_tree.csv")
print(f"Categories shape: {categories.shape}")
categories.head()

## 2. Basic Statistics

In [ ]:
print("=== Events Summary ===")
print(f"Total events:     {len(events):,}")
print(f"Unique visitors:  {events['visitorid'].nunique():,}")
print(f"Unique items:     {events['itemid'].nunique():,}")
print(f"Date range:       {pd.to_datetime(events['timestamp'], unit='ms').min()} to {pd.to_datetime(events['timestamp'], unit='ms').max()}")
print()
print(events.describe())

In [ ]:
print("\nNull counts:")
print(events.isnull().sum())
print(f"\nNull % in transactionid: {events['transactionid'].isnull().mean()*100:.1f}%")

## 3. Event Type Distribution

In [ ]:
event_counts = events['event'].value_counts()
print(event_counts)
print(f"\nPercentages:")
print((event_counts / len(events) * 100).round(2))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50']
event_counts.plot(kind='bar', color=colors, ax=ax, edgecolor='black')
ax.set_title('Event Type Distribution', fontsize=14)
ax.set_xlabel('Event Type')
ax.set_ylabel('Count')
for i, v in enumerate(event_counts):
    ax.text(i, v + len(events)*0.01, f'{v:,}', ha='center', fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_event_distribution.png'), dpi=150)
plt.show()

## 4. User Activity Distribution

In [ ]:
user_activity = events.groupby('visitorid').size()
print(f"Interactions per user:")
print(user_activity.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(user_activity.clip(upper=50), bins=50, color='#2196F3', edgecolor='black', alpha=0.8)
axes[0].set_title('Interactions per User (clipped at 50)', fontsize=12)
axes[0].set_xlabel('Number of Interactions')
axes[0].set_ylabel('Number of Users')

# Log-scale
axes[1].hist(user_activity, bins=100, color='#FF9800', edgecolor='black', alpha=0.8)
axes[1].set_yscale('log')
axes[1].set_xscale('log')
axes[1].set_title('Interactions per User (log-log scale)', fontsize=12)
axes[1].set_xlabel('Number of Interactions')
axes[1].set_ylabel('Number of Users (log)')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_user_activity.png'), dpi=150)
plt.show()

## 5. Item Popularity Distribution

In [ ]:
item_popularity = events.groupby('itemid').size().sort_values(ascending=False)
print(f"Interactions per item:")
print(item_popularity.describe())
print(f"\nTop 10 items:")
print(item_popularity.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(item_popularity)), item_popularity.values, color='#4CAF50', linewidth=0.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Item Popularity Distribution (log-log)', fontsize=14)
ax.set_xlabel('Item Rank (log)')
ax.set_ylabel('Number of Interactions (log)')
ax.axhline(y=3, color='red', linestyle='--', alpha=0.7, label='Min item threshold (3)')
ax.legend()
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_item_popularity.png'), dpi=150)
plt.show()

## 6. Temporal Trends

In [ ]:
events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['datetime'].dt.date

daily = events.groupby(['date', 'event']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
for col, color in zip(daily.columns, ['#2196F3', '#FF9800', '#4CAF50']):
    ax.plot(daily.index, daily[col], label=col, color=color, linewidth=1.5)
ax.set_title('Daily Events Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Event Count')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_temporal_trends.png'), dpi=150)
plt.show()

## 7. Sparsity Analysis

In [ ]:
n_users = events['visitorid'].nunique()
n_items = events['itemid'].nunique()
n_interactions = len(events)
possible_interactions = n_users * n_items
sparsity = 1 - (n_interactions / possible_interactions)

print(f"Users:                {n_users:,}")
print(f"Items:                {n_items:,}")
print(f"Interactions:         {n_interactions:,}")
print(f"Possible pairs:       {possible_interactions:,}")
print(f"Sparsity:             {sparsity*100:.4f}%")
print(f"Density:              {(1-sparsity)*100:.6f}%")

## 8. Category Analysis

In [ ]:
# Get categoryid from item properties
cat_props = item_props[item_props['property'] == 'categoryid'].copy()
cat_props['value'] = pd.to_numeric(cat_props['value'], errors='coerce')
item_cats = cat_props.drop_duplicates(subset='itemid', keep='last')[['itemid', 'value']]
item_cats.columns = ['itemid', 'categoryid']

# Merge with events
events_with_cat = events.merge(item_cats, on='itemid', how='left')
cat_dist = events_with_cat['categoryid'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
cat_dist.plot(kind='bar', color='#9C27B0', edgecolor='black', alpha=0.8, ax=ax)
ax.set_title('Top 20 Categories by Event Count', fontsize=14)
ax.set_xlabel('Category ID')
ax.set_ylabel('Number of Events')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_categories.png'), dpi=150)
plt.show()

## 9. Conversion Funnel

In [ ]:
event_counts_all = events['event'].value_counts()
views = event_counts_all.get('view', 0)
carts = event_counts_all.get('addtocart', 0)
txns = event_counts_all.get('transaction', 0)

print(f"Views:         {views:>10,}")
print(f"Add to cart:   {carts:>10,}  ({carts/views*100:.2f}% of views)")
print(f"Transactions:  {txns:>10,}  ({txns/views*100:.2f}% of views, {txns/carts*100:.2f}% of carts)")

fig, ax = plt.subplots(figsize=(8, 5))
stages = ['View', 'Add to Cart', 'Transaction']
values = [views, carts, txns]
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax.barh(stages[::-1], values[::-1], color=colors[::-1], edgecolor='black')
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + views*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_title('Conversion Funnel', fontsize=14)
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'eda_funnel.png'), dpi=150)
plt.show()

## 10. Summary

Key findings:
- The dataset is highly sparse (>99.99%), typical for e-commerce.
- Views dominate; conversion rates (view → cart → purchase) are low, as expected.
- User activity follows a power-law distribution (most users have few interactions).
- Item popularity also follows a long-tail distribution.
- Cold-start filtering (min 5 user interactions, min 3 item interactions) will reduce noise.
- Pseudo-rating scheme (view=1, cart=2, txn=3) captures implicit signal strength.

In [ ]:
print("EDA complete.")